# 10 — Production ML Pipeline

## Objective

This notebook converts the experimental uplift modeling workflow into a reproducible machine-learning pipeline.

The pipeline will:

- Load the Criteo dataset
- Create a reproducible development sample
- Select pre-treatment features
- Split treatment and control groups
- Train the selected uplift model
- Generate treatment-effect predictions
- Evaluate the model
- Save trained model artifacts
- Save model metadata
- Save evaluation results

The goal is to create a repeatable workflow that can later be moved from notebook experimentation into Python scripts and deployment infrastructure.

### Production Workflow

Raw Data
↓
Data Preparation
↓
Feature Selection
↓
Model Training
↓
Uplift Prediction
↓
Evaluation
↓
Model Artifact
↓
Production Deployment

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

import joblib

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
PROJECT_ROOT = Path(
    r"C:\Users\ugand\customer-churn-uplift-modeling"
)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "criteo-research-uplift-v2.1.csv.gz"
)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

MODELS_DIR = (
    PROJECT_ROOT
    / "models"
)

REPORTS_DIR = (
    PROJECT_ROOT
    / "reports"
)

print("Project root:", PROJECT_ROOT)
print("Dataset exists:", DATA_PATH.exists())

print("\nPath status:")

for path in [
    PROCESSED_DIR,
    MODELS_DIR,
    REPORTS_DIR
]:
    print(
        path,
        "->",
        "DIRECTORY" if path.is_dir()
        else "FILE" if path.is_file()
        else "MISSING"
    )

Project root: C:\Users\ugand\customer-churn-uplift-modeling
Dataset exists: True

Path status:
C:\Users\ugand\customer-churn-uplift-modeling\data\processed -> DIRECTORY
C:\Users\ugand\customer-churn-uplift-modeling\models -> FILE
C:\Users\ugand\customer-churn-uplift-modeling\reports -> FILE


In [4]:
RANDOM_STATE = 42

SAMPLE_SIZE = 100_000

CHUNK_SIZE = 250_000

ROWS_PER_CHUNK = 10_000

TEST_SIZE = 0.20

N_ESTIMATORS = 200

MAX_DEPTH = 10

MIN_SAMPLES_LEAF = 20

CLASS_WEIGHT = "balanced"

print("Configuration loaded successfully.")

Configuration loaded successfully.


In [5]:
FEATURE_COLS = [
    f"f{i}"
    for i in range(12)
]

TREATMENT_COL = "treatment"

OUTCOME_COL = "conversion"

print("Features:")
print(FEATURE_COLS)

Features:
['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11']


In [6]:
def load_development_sample(
    data_path,
    sample_size=100_000,
    chunk_size=250_000,
    rows_per_chunk=10_000,
    random_state=42
):
    
    sample_parts = []
    
    for chunk_number, chunk in enumerate(
        pd.read_csv(
            data_path,
            compression="gzip",
            chunksize=chunk_size
        )
    ):
        
        chunk_random_state = (
            random_state + chunk_number
        )
        
        current_sample_size = min(
            rows_per_chunk,
            len(chunk)
        )
        
        sampled_chunk = chunk.sample(
            n=current_sample_size,
            random_state=chunk_random_state
        )
        
        sample_parts.append(
            sampled_chunk
        )
    
    sample = pd.concat(
        sample_parts,
        ignore_index=True
    )
    
    if len(sample) > sample_size:
        sample = sample.sample(
            n=sample_size,
            random_state=random_state
        )
    
    return sample.reset_index(
        drop=True
    )


print("Sample loading function created.")

Sample loading function created.


In [7]:
df = load_development_sample(
    data_path=DATA_PATH,
    sample_size=SAMPLE_SIZE,
    chunk_size=CHUNK_SIZE,
    rows_per_chunk=ROWS_PER_CHUNK,
    random_state=RANDOM_STATE
)

print(
    "Development sample shape:",
    df.shape
)

Development sample shape: (100000, 16)


In [8]:
required_columns = (
    FEATURE_COLS
    + [
        TREATMENT_COL,
        OUTCOME_COL
    ]
)

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

print(
    "Missing columns:",
    missing_columns
)

assert len(missing_columns) == 0, (
    "Required columns are missing."
)

print(
    "Input data validation passed."
)

Missing columns: []
Input data validation passed.


In [9]:
X = df[
    FEATURE_COLS
].copy()

treatment = df[
    TREATMENT_COL
].copy()

y = df[
    OUTCOME_COL
].copy()

print("Feature matrix:", X.shape)
print("Treatment:", treatment.shape)
print("Outcome:", y.shape)

Feature matrix: (100000, 12)
Treatment: (100000,)
Outcome: (100000,)


In [10]:
assert set(
    treatment.unique()
) == {0, 1}, (
    "Treatment must contain both 0 and 1."
)

assert set(
    y.unique()
).issubset({0, 1}), (
    "Outcome must be binary."
)

print(
    "Treatment counts:"
)

print(
    treatment.value_counts()
    .sort_index()
)

print(
    "\nOutcome counts:"
)

print(
    y.value_counts()
    .sort_index()
)

Treatment counts:
treatment
0    15054
1    84946
Name: count, dtype: int64

Outcome counts:
conversion
0    99703
1      297
Name: count, dtype: int64


In [11]:
X_train, X_test, y_train, y_test, treatment_train, treatment_test = (
    train_test_split(
        X,
        y,
        treatment,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y
    )
)

print(
    "Training rows:",
    len(X_train)
)

print(
    "Test rows:",
    len(X_test)
)

Training rows: 80000
Test rows: 20000


In [12]:
control_mask = (
    treatment_train == 0
)

treatment_mask = (
    treatment_train == 1
)

X_control = X_train.loc[
    control_mask
].copy()

y_control = y_train.loc[
    control_mask
].copy()

X_treatment = X_train.loc[
    treatment_mask
].copy()

y_treatment = y_train.loc[
    treatment_mask
].copy()

print(
    "Control samples:",
    len(X_control)
)

print(
    "Treatment samples:",
    len(X_treatment)
)

Control samples: 12036
Treatment samples: 67964


In [13]:
control_model = RandomForestClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    class_weight=CLASS_WEIGHT,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

treatment_model = RandomForestClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    class_weight=CLASS_WEIGHT,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print("Production model objects created.")

Production model objects created.


In [14]:
control_model.fit(
    X_control,
    y_control
)

print(
    "Control model trained successfully."
)

Control model trained successfully.


In [15]:
treatment_model.fit(
    X_treatment,
    y_treatment
)

print(
    "Treatment model trained successfully."
)

Treatment model trained successfully.


In [16]:
predicted_control = (
    control_model
    .predict_proba(X_test)[:, 1]
)

predicted_treatment = (
    treatment_model
    .predict_proba(X_test)[:, 1]
)

predicted_uplift = (
    predicted_treatment -
    predicted_control
)

print(
    "Predictions generated successfully."
)

print(
    "Mean predicted uplift:",
    predicted_uplift.mean()
)

Predictions generated successfully.
Mean predicted uplift: 0.013582217159636431


In [17]:
production_predictions = pd.DataFrame({
    "customer_index": X_test.index,
    "actual_treatment": treatment_test.values,
    "actual_conversion": y_test.values,
    "predicted_control": predicted_control,
    "predicted_treatment": predicted_treatment,
    "predicted_uplift": predicted_uplift
})

production_predictions.head()

,customer_index,actual_treatment,actual_conversion,predicted_control,predicted_treatment,predicted_uplift
0,87560,1,0,0.000491,0.003072,0.002581
1,22450,1,0,0.095366,0.011081,-0.084285
2,98559,1,0,0.000491,0.004170,0.003679
3,58708,1,0,0.000491,0.013472,0.012981
4,38520,1,0,0.005904,0.201912,0.196007


In [18]:
observed_prediction = np.where(
    treatment_test.values == 1,
    predicted_treatment,
    predicted_control
)

roc_auc = roc_auc_score(
    y_test,
    observed_prediction
)

pr_auc = average_precision_score(
    y_test,
    observed_prediction
)

print(
    f"ROC-AUC: {roc_auc:.6f}"
)

print(
    f"PR-AUC: {pr_auc:.6f}"
)

ROC-AUC: 0.940856
PR-AUC: 0.178952


In [19]:
def uplift_at_k(
    uplift_scores,
    fraction=0.10
):
    
    n = int(
        len(uplift_scores) *
        fraction
    )
    
    if n == 0:
        return np.nan
    
    ranked_scores = np.sort(
        uplift_scores
    )[::-1]
    
    return ranked_scores[:n].mean()


uplift_10 = uplift_at_k(
    predicted_uplift,
    0.10
)

uplift_20 = uplift_at_k(
    predicted_uplift,
    0.20
)

print(
    f"Uplift@10%: {uplift_10:.6f}"
)

print(
    f"Uplift@20%: {uplift_20:.6f}"
)

Uplift@10%: 0.149753
Uplift@20%: 0.091474


In [20]:
production_metrics = pd.DataFrame({
    "metric": [
        "ROC-AUC",
        "PR-AUC",
        "Uplift@10%",
        "Uplift@20%",
        "Mean Predicted Uplift"
    ],
    "value": [
        roc_auc,
        pr_auc,
        uplift_10,
        uplift_20,
        predicted_uplift.mean()
    ]
})

production_metrics

,metric,value
0,ROC-AUC,0.940856
1,PR-AUC,0.178952
2,Uplift@10%,0.149753
3,Uplift@20%,0.091474
4,Mean Predicted Uplift,0.013582


In [21]:
production_predictions = (
    production_predictions
    .sort_values(
        "predicted_uplift",
        ascending=False
    )
    .reset_index(drop=True)
)

production_predictions["rank"] = (
    np.arange(
        len(production_predictions)
    ) + 1
)

production_predictions["treatment_recommendation"] = np.where(
    production_predictions["predicted_uplift"] > 0,
    "Treat",
    "Do Not Treat"
)

production_predictions.head(20)

,customer_index,actual_treatment,actual_conversion,predicted_control,predicted_treatment,predicted_uplift,rank,treatment_recommendation
0,7084,1,0,0.127235,0.713814,0.586579,1,Treat
1,10735,1,0,0.179967,0.761409,0.581442,2,Treat
2,4708,1,0,0.217890,0.791669,0.573779,3,Treat
3,54439,1,0,0.101021,0.662938,0.561917,4,Treat
4,49859,1,0,0.343001,0.903500,0.560499,5,Treat
5,84013,0,0,0.163081,0.719748,0.556667,6,Treat
6,21251,0,0,0.227794,0.777863,0.550069,7,Treat
7,74505,0,0,0.343765,0.892040,0.548275,8,Treat
8,20163,0,0,0.232318,0.772666,0.540348,9,Treat
9,69005,1,0,0.305721,0.844648,0.538928,10,Treat


In [22]:
prediction_output_path = (
    PROCESSED_DIR /
    "production_uplift_predictions.csv"
)

production_predictions.to_csv(
    prediction_output_path,
    index=False
)

print(
    "Predictions saved to:"
)

print(
    prediction_output_path
)

Predictions saved to:
C:\Users\ugand\customer-churn-uplift-modeling\data\processed\production_uplift_predictions.csv


In [27]:
from pathlib import Path

MODELS_FILE = Path(
    r"C:\Users\ugand\customer-churn-uplift-modeling\models"
)

print("Exists:", MODELS_FILE.exists())
print("Is file:", MODELS_FILE.is_file())
print("Is directory:", MODELS_FILE.is_dir())
print("Size:", MODELS_FILE.stat().st_size, "bytes")

Exists: True
Is file: True
Is directory: False
Size: 2 bytes


In [28]:
BACKUP_PATH = Path(
    r"C:\Users\ugand\customer-churn-uplift-modeling\models_old"
)

MODELS_FILE.rename(BACKUP_PATH)

print("Existing 'models' file renamed to:")
print(BACKUP_PATH)

Existing 'models' file renamed to:
C:\Users\ugand\customer-churn-uplift-modeling\models_old


In [29]:
MODELS_DIR = Path(
    r"C:\Users\ugand\customer-churn-uplift-modeling\models"
)

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Models directory created successfully.")

Models directory created successfully.


In [30]:
control_model_path = (
    MODELS_DIR /
    "t_learner_control_model.joblib"
)

treatment_model_path = (
    MODELS_DIR /
    "t_learner_treatment_model.joblib"
)

joblib.dump(
    control_model,
    control_model_path
)

joblib.dump(
    treatment_model,
    treatment_model_path
)

print("Control model saved to:")
print(control_model_path)

print("\nTreatment model saved to:")
print(treatment_model_path)

Control model saved to:
C:\Users\ugand\customer-churn-uplift-modeling\models\t_learner_control_model.joblib

Treatment model saved to:
C:\Users\ugand\customer-churn-uplift-modeling\models\t_learner_treatment_model.joblib


In [31]:
# Verify saved model files

print("Control model exists:", control_model_path.exists())
print("Treatment model exists:", treatment_model_path.exists())

print("\nControl model size:",
      round(control_model_path.stat().st_size / (1024 * 1024), 2), "MB")

print("Treatment model size:",
      round(treatment_model_path.stat().st_size / (1024 * 1024), 2), "MB")

Control model exists: True
Treatment model exists: True

Control model size: 0.57 MB
Treatment model size: 2.61 MB


In [32]:
loaded_control_model = joblib.load(
    control_model_path
)

loaded_treatment_model = joblib.load(
    treatment_model_path
)

print(
    "Saved models loaded successfully."
)

Saved models loaded successfully.


In [33]:
loaded_control_prediction = (
    loaded_control_model
    .predict_proba(X_test)[:, 1]
)

loaded_treatment_prediction = (
    loaded_treatment_model
    .predict_proba(X_test)[:, 1]
)

loaded_uplift = (
    loaded_treatment_prediction -
    loaded_control_prediction
)

print(
    "Maximum prediction difference:"
)

print(
    np.max(
        np.abs(
            predicted_uplift -
            loaded_uplift
        )
    )
)

Maximum prediction difference:
5.551115123125783e-16


In [34]:
model_metadata = {
    "model_type": "T-Learner",
    "base_estimator": "RandomForestClassifier",
    "feature_columns": FEATURE_COLS,
    "outcome_column": OUTCOME_COL,
    "treatment_column": TREATMENT_COL,
    "sample_size": len(df),
    "training_size": len(X_train),
    "test_size": len(X_test),
    "n_estimators": N_ESTIMATORS,
    "max_depth": MAX_DEPTH,
    "min_samples_leaf": MIN_SAMPLES_LEAF,
    "class_weight": CLASS_WEIGHT,
    "random_state": RANDOM_STATE,
    "roc_auc": float(roc_auc),
    "pr_auc": float(pr_auc),
    "uplift_at_10": float(uplift_10),
    "uplift_at_20": float(uplift_20),
    "created_at": datetime.now().isoformat()
}

model_metadata

{'model_type': 'T-Learner',
 'base_estimator': 'RandomForestClassifier',
 'feature_columns': ['f0',
  'f1',
  'f2',
  'f3',
  'f4',
  'f5',
  'f6',
  'f7',
  'f8',
  'f9',
  'f10',
  'f11'],
 'outcome_column': 'conversion',
 'treatment_column': 'treatment',
 'sample_size': 100000,
 'training_size': 80000,
 'test_size': 20000,
 'n_estimators': 200,
 'max_depth': 10,
 'min_samples_leaf': 20,
 'class_weight': 'balanced',
 'random_state': 42,
 'roc_auc': 0.9408564587567221,
 'pr_auc': 0.1789522257172882,
 'uplift_at_10': 0.14975265394948428,
 'uplift_at_20': 0.09147406775382731,
 'created_at': '2026-09-06T09:31:28.195474'}

In [35]:
import json

metadata_path = (
    MODELS_DIR /
    "t_learner_model_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as file:
    
    json.dump(
        model_metadata,
        file,
        indent=4
    )

print(
    "Metadata saved to:"
)

print(
    metadata_path
)

Metadata saved to:
C:\Users\ugand\customer-churn-uplift-modeling\models\t_learner_model_metadata.json


In [39]:
from pathlib import Path

REPORTS_PATH = Path(
    r"C:\Users\ugand\customer-churn-uplift-modeling\reports"
)

print("Exists:", REPORTS_PATH.exists())
print("Is file:", REPORTS_PATH.is_file())
print("Is directory:", REPORTS_PATH.is_dir())

Exists: True
Is file: True
Is directory: False


In [40]:
BACKUP_REPORTS = Path(
    r"C:\Users\ugand\customer-churn-uplift-modeling\reports_old"
)

REPORTS_PATH.rename(BACKUP_REPORTS)

print("Existing reports file renamed to:")
print(BACKUP_REPORTS)

Existing reports file renamed to:
C:\Users\ugand\customer-churn-uplift-modeling\reports_old


In [41]:
REPORTS_DIR = Path(
    r"C:\Users\ugand\customer-churn-uplift-modeling\reports"
)

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Reports directory created successfully:")
print(REPORTS_DIR)

Reports directory created successfully:
C:\Users\ugand\customer-churn-uplift-modeling\reports


In [42]:
metrics_path = REPORTS_DIR / "production_model_metrics.csv"

production_metrics.to_csv(
    metrics_path,
    index=False
)

print("Metrics saved to:")
print(metrics_path)

Metrics saved to:
C:\Users\ugand\customer-churn-uplift-modeling\reports\production_model_metrics.csv


In [43]:
print("Metrics file exists:", metrics_path.exists())
print("File size:", round(metrics_path.stat().st_size / 1024, 2), "KB")

Metrics file exists: True
File size: 0.17 KB


In [45]:
from pathlib import Path

PROCESSED_DIR = Path(
    r"C:\Users\ugand\customer-churn-uplift-modeling\data\processed"
)

print("Production prediction files:")

for file in PROCESSED_DIR.glob("*production*"):
    print(file)

Production prediction files:
C:\Users\ugand\customer-churn-uplift-modeling\data\processed\production_uplift_predictions.csv


In [46]:
production_predictions_path = (
    PROCESSED_DIR / "production_uplift_predictions.csv"
)

print("Exists:", production_predictions_path.exists())
print(production_predictions_path)

Exists: True
C:\Users\ugand\customer-churn-uplift-modeling\data\processed\production_uplift_predictions.csv


In [47]:
checks = {
    "Control model exists": control_model_path.exists(),
    "Treatment model exists": treatment_model_path.exists(),
    "Production predictions exist": production_predictions_path.exists(),
    "Metrics report exists": metrics_path.exists(),
    "Metadata exists": metadata_path.exists(),
}

for check, status in checks.items():
    print(f"{check}: {'PASS' if status else 'FAIL'}")

Control model exists: PASS
Treatment model exists: PASS
Production predictions exist: PASS
Metrics report exists: PASS
Metadata exists: PASS


In [48]:
print("\nProduction Model Metrics:")
display(production_metrics)

print("\nSample Production Predictions:")
display(production_predictions.head())


Production Model Metrics:


,metric,value
0,ROC-AUC,0.940856
1,PR-AUC,0.178952
2,Uplift@10%,0.149753
3,Uplift@20%,0.091474
4,Mean Predicted Uplift,0.013582



Sample Production Predictions:


,customer_index,actual_treatment,actual_conversion,predicted_control,predicted_treatment,predicted_uplift,rank,treatment_recommendation
0,7084,1,0,0.127235,0.713814,0.586579,1,Treat
1,10735,1,0,0.179967,0.761409,0.581442,2,Treat
2,4708,1,0,0.217890,0.791669,0.573779,3,Treat
3,54439,1,0,0.101021,0.662938,0.561917,4,Treat
4,49859,1,0,0.343001,0.903500,0.560499,5,Treat


In [ ]:
artifact_summary = pd.DataFrame({
    "artifact": [
        "Control model",
        "Treatment model",
        "Model metadata",
        "Production predictions",
        "Production metrics"
    ],
    "path": [
        str(control_model_path),
        str(treatment_model_path),
        str(metadata_path),
        str(prediction_output_path),
        str(metrics_path)
    ]
})

artifact_summary

# Conclusion

In this notebook, the uplift modeling workflow was converted into a reproducible **production-oriented machine learning pipeline** using the T-Learner approach.

The main outcomes were:

- Created a reproducible data loading and sampling workflow.
- Validated the required dataset columns and treatment/outcome variables.
- Created a stratified train-test split.
- Trained separate Random Forest models for the treatment and control groups.
- Generated treatment and control outcome probabilities.
- Calculated individual predicted uplift as:

  `P(conversion | treatment, X) - P(conversion | control, X)`

- Evaluated the production model using ROC-AUC, PR-AUC, and uplift-based diagnostics.
- Generated production uplift predictions.
- Saved the trained treatment and control models as `.joblib` artifacts.
- Verified that the saved models could be loaded successfully.
- Verified prediction reproducibility after model loading.
- Saved model metadata and production evaluation metrics.
- Performed final production-readiness checks.

The resulting production workflow is:

Raw Dataset
→ Data Sampling
→ Train/Test Split
→ Treatment & Control Models
→ Uplift Prediction
→ Model Evaluation
→ Model Artifacts
→ Production Predictions
→ Metadata & Metrics
→ Readiness Checks

This notebook establishes a reproducible baseline for taking the uplift model from experimentation toward deployment.

The trained model artifacts, production predictions, metadata, and evaluation metrics are stored within the project structure so that they can be tracked and reused by downstream MLOps components.

The next stage extends this workflow with **experiment tracking and model lifecycle management using MLflow**, enabling systematic tracking of model parameters, metrics, artifacts, and model versions.